In [1]:
from pyspark import SparkContext

logFile = "data/lotr.txt"  # Should be some file on your system
sc = SparkContext("local", "Simple App")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/17 14:28:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/17 14:28:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
logData = sc.textFile(logFile).cache()

numAs = logData.filter(lambda s: 'a' in s).count()
numBs = logData.filter(lambda s: 'b' in s).count()

print("Lines with a: %i, lines with b: %i" % (numAs, numBs))

Lines with a: 182, lines with b: 155


# products.txt

In [13]:
import os

products_file = os.path.join('data', 'products.txt')

In [14]:
prod_raw = sc.textFile(products_file).cache()
prod_raw.take(3)

['1\tapple\t3.3\tfruit', '2\torange\t6.5\tfruit', '3\tbeer\t10\tbeverage']

In [ ]:
def format(line):
    line = line.split('\t')
    line[0] = int(line[0])
    line[2] = float(line[2])
    return line

prod = prod_raw.map(format)

prod.collect()

[[1, 'apple', 3.3, 'fruit'],
 [2, 'orange', 6.5, 'fruit'],
 [3, 'beer', 10.0, 'beverage'],
 [4, 'wine', 15.0, 'beverage'],
 [5, 'TAOCP1', 20.9, 'book'],
 [6, 'TAOCP2', 5.2, 'book'],
 [7, 'film1', 30.0, 'film'],
 [8, 'film2', 20.4, 'film'],
 [9, 'film3', 33.0, 'film'],
 [10, 'film4', 42.9, 'film'],
 [11, 'film5', 13.3, 'film'],
 [12, 'milk', 7.6, 'beverage'],
 [13, 'banana', 5.3, 'fruit'],
 [14, 'grapes', 7.5, 'fruit'],
 [15, 'soda', 4.0, 'beverage'],
 [16, 'water', 2.0, 'beverage'],
 [17, 's_water', 3.5, 'beverage'],
 [18, 'grappe', 8.2, 'fruit'],
 [19, 'pear', 7.0, 'fruit'],
 [20, 'strawberry', 9.0, 'fruit'],
 [21, 'plum', 4.4, 'fruit'],
 [22, 'mango', 17.5, 'fruit']]

In [20]:
prod20 = prod.filter(lambda x: x[2]>20)
prod20.collect()

[[5, 'TAOCP1', 20.9, 'book'],
 [7, 'film1', 30.0, 'film'],
 [8, 'film2', 20.4, 'film'],
 [9, 'film3', 33.0, 'film'],
 [10, 'film4', 42.9, 'film']]

In [39]:
prod20.sortBy(lambda t: t[2])
prod20.collect()

[[5, 'TAOCP1', 20.9, 'book'],
 [7, 'film1', 30.0, 'film'],
 [8, 'film2', 20.4, 'film'],
 [9, 'film3', 33.0, 'film'],
 [10, 'film4', 42.9, 'film']]

In [42]:
prod_by_category = prod.map(lambda x: (x[3], (x[0], x[1], x[2])))

maxpricebycat = prod_by_category.reduceByKey(lambda a,b: a if (a[2]>b[2]) else b)
maxpricebycat.collect()

[('fruit', (22, 'mango', 17.5)),
 ('beverage', (4, 'wine', 15.0)),
 ('book', (5, 'TAOCP1', 20.9)),
 ('film', (10, 'film4', 42.9))]

In [45]:
countbycat = prod_by_category.map(lambda x: (x[0], 1))
countbycat = countbycat.reduceByKey(lambda a,b: a+b)
countbycat.collect()

[('fruit', 9), ('beverage', 6), ('book', 2), ('film', 5)]

In [47]:
def avg(a,b):
    return 0.5*(a+b)

wrong_avg = prod_by_category.map(lambda x: (x[0], x[1][2]))
wrong_avg = wrong_avg.reduceByKey(avg)

wrong_avg.collect()

[('fruit', 11.865625),
 ('beverage', 4.00625),
 ('book', 13.049999999999999),
 ('film', 24.65)]

In [49]:
wrong_avg_sorted = prod_by_category.map(lambda x: (x[0], x[1][2])).sortBy(lambda t: t[1])
wrong_avg_sorted = wrong_avg_sorted.reduceByKey(avg)

wrong_avg_sorted.collect()

[('beverage', 11.371875),
 ('fruit', 12.885546875),
 ('book', 13.049999999999999),
 ('film', 35.55625)]

In [ ]:
totals = prod_by_category.map(lambda x: (x[0], (x[1][2], 1)))
totals = totals.reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1]))

avg_price = totals.map(lambda x: (x[0], x[1][0]/x[1][1]))

avg_price.collect()

[('fruit', 7.633333333333332),
 ('beverage', 7.016666666666667),
 ('book', 13.049999999999999),
 ('film', 27.920000000000005)]